# Fixmatch-based Semisupervised Subclassifier of Myeloid

## Create Croppings

In [ ]:
from ultralytics import YOLO
import cv2
from pathlib import Path

# === Configuration ===
model_path = "/Users/user/runs/detect/train11/weights/best.pt"
image_dir = Path("/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/Normalpics")
output_dir = Path("/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells")
conf_threshold = 0.8  # Adjust as needed
image_extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG")

# === Prepare Directories ===
output_dir.mkdir(parents=True, exist_ok=True)

# === Load YOLO Model ===
model = YOLO(model_path)

# === Process Each Image ===
for ext in image_extensions:
    for img_path in image_dir.glob(ext):
        results = model(str(img_path))[0]
        img = results.orig_img  # Already in BGR from OpenCV internally
        h, w = img.shape[:2]

        for i, box in enumerate(results.boxes):
            cls_id = int(box.cls.item())
            conf = box.conf.item()
            class_name = model.names[cls_id]

            # Get bounding box coordinates
            xmin, ymin, xmax, ymax = map(int, box.xyxy.cpu().numpy()[0])

            # Apply confidence threshold
            if conf < conf_threshold:
                continue

            # Clamp to image bounds
            xmin, ymin = max(0, xmin), max(0, ymin)
            xmax, ymax = min(w, xmax), min(h, ymax)

            # Skip invalid boxes
            if xmin >= xmax or ymin >= ymax:
                continue

            # Crop the detected region (already in BGR)
            cropped = img[ymin:ymax, xmin:xmax]

            # Save to class-named folder
            class_dir = output_dir / class_name
            class_dir.mkdir(exist_ok=True, parents=True)

            save_path = class_dir / f"{img_path.stem}_{i}_{conf:.2f}_{class_name}.jpg"
            cv2.imwrite(str(save_path), cropped)

print("✅ Cropping completed successfully!")

## Building Fixmatch Efficientnet B0

In [ ]:
import os
import shutil
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from datetime import datetime

# === 🔧 Customize Your Output Paths ===
MODEL_SAVE_PATH = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Subclassing_Myeloid/marrows_fixmatch_model.pth"
PSEUDO_LABEL_DIR = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Subclassing_Myeloid/pseudo_labels"

# ========= Augmentations =========
class WeakAug:
    def __call__(self, x):
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])(x)

class StrongAug:
    def __call__(self, x):
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.RandomAffine(degrees=20, translate=(0.1, 0.1)),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])(x)

# ========= Unlabeled Dataset with Export Paths =========
class UnlabeledDataset(Dataset):
    def __init__(self, root):
        self.paths = [os.path.join(root, f) for f in os.listdir(root) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
        self.weak_aug = WeakAug()
        self.strong_aug = StrongAug()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = Image.open(img_path).convert('RGB')
        return self.weak_aug(img), self.strong_aug(img), img_path

# ========= Dataloaders =========
def get_dataloaders(labeled_path, unlabeled_path, batch_size):
    labeled_dataset = ImageFolder(labeled_path, transform=WeakAug())
    class_names = labeled_dataset.classes
    unlabeled_dataset = UnlabeledDataset(unlabeled_path)

    labeled_loader = DataLoader(labeled_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    return labeled_loader, unlabeled_loader, class_names

# ========= EfficientNet Model =========
def get_model(num_classes):
    model = models.efficientnet_b0(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

# ========= FixMatch Training =========
def train_fixmatch(model, labeled_loader, unlabeled_loader, class_names, device, epochs=20, threshold=0.95):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    model.train()

    # Create pseudo-label export directory
    os.makedirs(PSEUDO_LABEL_DIR, exist_ok=True)
    for cname in class_names:
        os.makedirs(os.path.join(PSEUDO_LABEL_DIR, cname), exist_ok=True)

    unlabeled_iter = iter(unlabeled_loader)

    for epoch in range(epochs):
        total_loss = 0
        for (x_lb, y_lb) in tqdm(labeled_loader, desc=f"Epoch {epoch+1}"):
            try:
                x_ul_w, x_ul_s, img_paths = next(unlabeled_iter)
            except StopIteration:
                unlabeled_iter = iter(unlabeled_loader)
                x_ul_w, x_ul_s, img_paths = next(unlabeled_iter)

            x_lb, y_lb = x_lb.to(device), y_lb.to(device)
            x_ul_w, x_ul_s = x_ul_w.to(device), x_ul_s.to(device)

            # --- Pseudo-labeling ---
            with torch.no_grad():
                logits_ul_w = model(x_ul_w)
                probs = torch.softmax(logits_ul_w, dim=1)
                max_probs, pseudo_labels = torch.max(probs, dim=1)
                mask = max_probs.ge(threshold).float()

            # --- Supervised loss ---
            logits_lb = model(x_lb)
            loss_sup = criterion(logits_lb, y_lb)

            # --- Unsupervised loss ---
            logits_ul_s = model(x_ul_s)
            loss_unsup = (criterion(logits_ul_s, pseudo_labels) * mask).mean()

            loss = loss_sup + loss_unsup
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # --- Save confident pseudo-labeled images ---
            for i in range(len(mask)):
                if mask[i] == 1.0:
                    pred_idx = pseudo_labels[i].item()
                    class_name = class_names[pred_idx]
                    dest_path = os.path.join(PSEUDO_LABEL_DIR, class_name, os.path.basename(img_paths[i]))
                    if not os.path.exists(dest_path):  # avoid overwrite
                        shutil.copy(img_paths[i], dest_path)

        print(f"Epoch {epoch+1}/{epochs} | Total Loss: {total_loss:.4f}")

    # === Save Trained Model ===
    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"\n✅ Trained model saved to: {MODEL_SAVE_PATH}")

# ========= Main =========
if __name__ == "__main__":
    labeled_path = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Myeloid_FixMatch/Labeled"
    unlabeled_path = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Myeloid_FixMatch/Unlabeled"
    batch_size = 16

    # Use MPS if on MacBook with Apple Silicon
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    labeled_loader, unlabeled_loader, class_names = get_dataloaders(labeled_path, unlabeled_path, batch_size)
    model = get_model(num_classes=len(class_names)).to(device)

    print(f"Detected classes: {class_names}")
    train_fixmatch(model, labeled_loader, unlabeled_loader, class_names, device)


#### Classes sequence: "Band", "Seg", "Basophils", "Monoblast", "Metamyelocyte", "Myelocyte", "Monocyte", "Progranulocyte", "Myeloblast"

# Check for Embeddings Performance: Cluster Analysis of the Labels and Silhouette Score

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import umap
from sklearn.metrics import silhouette_score
from tqdm import tqdm

# ==== Config ====
MODEL_PATH = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Subclassing_Myeloid/marrows_fixmatch_model.pth"
DATA_DIR = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Myeloid_FixMatch/Labeled"
BATCH_SIZE = 32
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

class_names = [
    "Band", "Basophils", "Metamyelocyte", "Monoblast",
    "Monocyte", "Myeloblast", "Myelocyte",
    "Progranulocyte", "Seg"
]
# ==== Load Model ====
base_model = models.efficientnet_b0(pretrained=False)
base_model.classifier[1] = nn.Linear(base_model.classifier[1].in_features, len(class_names))
base_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
base_model.to(device)
base_model.eval()

# ==== Transforms and Dataset ====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==== Extract Features ====
features = []
labels = []

with torch.no_grad():
    for imgs, lbls in tqdm(dataloader):
        imgs = imgs.to(device)
        feats = base_model.features(imgs)
        pooled = base_model.avgpool(feats)
        pooled = torch.flatten(pooled, 1)
        features.append(pooled.cpu().numpy())
        labels.append(lbls.numpy())

features = np.concatenate(features, axis=0)
labels = np.concatenate(labels, axis=0)

# ==== UMAP ====
print("Running UMAP...")
reducer = umap.UMAP(n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(features)

# ==== Compute Silhouette Score ====
sil_score = silhouette_score(embedding_2d, labels)
print(f"🔍 Silhouette Score: {sil_score:.3f}")

# ==== Compute Class Centroids ====
centroids = []
for i in range(len(class_names)):
    class_points = embedding_2d[labels == i]
    centroid = class_points.mean(axis=0)
    centroids.append(centroid)
centroids = np.array(centroids)

# ==== Plot ====
plt.figure(figsize=(12, 8))

colors = plt.cm.get_cmap("tab10", len(class_names))

for i, cname in enumerate(class_names):
    idxs = labels == i
    plt.scatter(embedding_2d[idxs, 0], embedding_2d[idxs, 1],
                label=cname, alpha=0.6, s=20, color=colors(i))

    # Centroid marker
    plt.scatter(centroids[i, 0], centroids[i, 1],
                color=colors(i), edgecolor="black", s=200, marker="X", linewidth=1.5)

plt.title(f"UMAP of FixMatch Embeddings — Silhouette Score = {sil_score:.3f}")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend()
plt.tight_layout()
plt.show()

## Shillouette Score: 0.77 = Good clustering

<img alt="UMAP Representation of the Embbeddings" height="300" src="/Users/user/Desktop/Kasus Sulid/Screenshot 2025-07-09 at 01.11.24.png" width="300"/>

##### Sample image hasil Fixmatch >>>> Vanilla Efficientnet, nah pas disuruh implement, hasilnya cukup burik jadi aku sekarang sedang eksplorasi sebelum mosaic augmentation

## Code for Implementing FixMatch Effi Model

In [ ]:
import os
import shutil
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms

# ==== Paths ====
MODEL_PATH = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Subclassing_Myeloid/marrows_fixmatch_model.pth"
UNLABELED_DIR = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Myeloid_FixMatch/Unlabeled"
OUTPUT_DIR = "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Subclassing_Myeloid/Pseudo_clean"
CONFIDENCE_THRESHOLD = 0.6

# ==== Class Order ====
class_names = [
    "Band", "Basophils", "Metamyelocyte", "Monoblast",
    "Monocyte", "Myeloblast", "Myelocyte",
    "Progranulocyte", "Seg"
]

# ==== Device ====
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"🔧 Using device: {device}")

# ==== Load Model ====
model = models.efficientnet_b0(pretrained=False)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# ==== Transform ====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ==== Create output folders ====
for cname in class_names:
    os.makedirs(os.path.join(OUTPUT_DIR, cname), exist_ok=True)

# ==== Predict + Save ====
with torch.no_grad():
    for fname in os.listdir(UNLABELED_DIR):
        if not fname.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        path = os.path.join(UNLABELED_DIR, fname)
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"⚠️ Could not open {fname}: {e}")
            continue

        x = transform(img).unsqueeze(0).to(device)
        output = model(x)
        probs = torch.softmax(output, dim=1)
        conf, pred_idx = torch.max(probs, dim=1)

        confidence = conf.item()
        pred_class = class_names[pred_idx.item()]

        if confidence >= CONFIDENCE_THRESHOLD:
            # Add confidence to filename
            orig_name, ext = os.path.splitext(fname)
            new_fname = f"{orig_name}_{confidence:.4f}.jpg"
            dst_path = os.path.join(OUTPUT_DIR, pred_class, new_fname)

            shutil.copy(path, dst_path)

print("✅ Pseudo-labels with confidence ≥ {:.2f} saved with confidence in filenames.".format(CONFIDENCE_THRESHOLD))
